In [ ]:
import sys, os, json, warnings
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('.'))
from vanilla_inpaint import ddpm_inpaint
from boundary_inpaint import ddpm_inpaint_boundary
from annealed_inpaint import ddpm_inpaint_annealed
from freq_inpaint import ddpm_inpaint_freq
from utils.cli import preprocess_inputs, load_sd_pipeline
from utils.image import apply_mask_for_display

In [ ]:
MY_IMAGES_DIR  = './our_images'
STEPS          = 50
GUIDANCE_SCALE = 7.5
SEED           = 42
SOFT_ZONE      = 12
GAMMA          = 0.5
LOW_ALPHA      = 0.5
HIGH_ALPHA     = 1.0
KERNEL_SIZE    = 5

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f'Using device: {device}')
pipe = load_sd_pipeline(device)
pipe.set_progress_bar_config(disable=True)
print('Model loaded.')

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

pairs = []
for fname in sorted(os.listdir(MY_IMAGES_DIR)):
    stem, ext = os.path.splitext(fname)
    if ext not in IMAGE_EXTENSIONS:
        continue
    if stem.endswith('_mask'):
        continue
    mask_path = os.path.join(MY_IMAGES_DIR, f"{stem}_mask.pt")
    if not os.path.exists(mask_path):
        print(f'  [skip] no mask for {fname}')
        continue
    json_path = os.path.join(MY_IMAGES_DIR, f"{stem}.json")
    caption = ''
    if os.path.exists(json_path):
        with open(json_path) as f:
            caption = json.load(f).get('caption', '')
    pairs.append({
        'img_path':  os.path.join(MY_IMAGES_DIR, fname),
        'mask_path': mask_path,
        'stem':      stem,
        'caption':   caption,
    })

print(f'Discovered {len(pairs)} image/mask pairs:\n')
for p in pairs:
    print(f"  {p['stem']:25s}  caption: {p['caption']}")

In [ ]:
out_dir = './output_novel_comparison'
os.makedirs(out_dir, exist_ok=True)

results = []

for p in tqdm(pairs, desc='Images'):
    prompt = p['caption']
    original_pil, mask = preprocess_inputs(p['img_path'], p['mask_path'])

    result_vanilla = ddpm_inpaint(
        pipe, original_pil, mask, prompt, STEPS, GUIDANCE_SCALE, SEED
    )
    result_boundary = ddpm_inpaint_boundary(
        pipe, original_pil, mask, prompt, STEPS, GUIDANCE_SCALE, SEED,
        soft_zone_pixels=SOFT_ZONE,
    )
    result_annealed = ddpm_inpaint_annealed(
        pipe, original_pil, mask, prompt, STEPS, GUIDANCE_SCALE, SEED,
        gamma=GAMMA,
    )
    result_freq = ddpm_inpaint_freq(
        pipe, original_pil, mask, prompt, STEPS, GUIDANCE_SCALE, SEED,
        low_alpha=LOW_ALPHA, high_alpha=HIGH_ALPHA, kernel_size=KERNEL_SIZE,
    )

    masked_viz = apply_mask_for_display(original_pil, mask)

    results.append({
        'stem':      p['stem'],
        'caption':   prompt,
        'original':  original_pil,
        'masked':    masked_viz,
        'vanilla':   result_vanilla,
        'boundary':  result_boundary,
        'annealed':  result_annealed,
        'freq':      result_freq,
    })

    # Save individual results
    result_vanilla.save(os.path.join(out_dir, f"{p['stem']}_vanilla.png"))
    result_boundary.save(os.path.join(out_dir, f"{p['stem']}_boundary.png"))
    result_annealed.save(os.path.join(out_dir, f"{p['stem']}_annealed.png"))
    result_freq.save(os.path.join(out_dir, f"{p['stem']}_freq.png"))

print(f'Done. Results saved to {out_dir}/')

In [ ]:
for r in results:
    fig, axes = plt.subplots(1, 6, figsize=(24, 4))
    fig.suptitle(f"{r['stem']}  —  \"{r['caption']}\"", fontsize=11)

    axes[0].imshow(r['original']);  axes[0].set_title('Original');   axes[0].axis('off')
    axes[1].imshow(r['masked']);    axes[1].set_title('Masked');     axes[1].axis('off')
    axes[2].imshow(r['vanilla']);   axes[2].set_title('Vanilla');    axes[2].axis('off')
    axes[3].imshow(r['boundary']);  axes[3].set_title('Boundary\nSoftening'); axes[3].axis('off')
    axes[4].imshow(r['annealed']);  axes[4].set_title('Annealed\nConstraint'); axes[4].axis('off')
    axes[5].imshow(r['freq']);      axes[5].set_title('Freq-Aware\nBlending'); axes[5].axis('off')

    plt.tight_layout()
    save_path = os.path.join(out_dir, f"{r['stem']}_comparison.png")
    fig.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")